# SDG Dry-Run: 10-Example Format Validation (D-19)

This notebook runs a minimal 10-example generation to validate the OSFT output format
before committing to the full production run (8,500-14,000 examples).

**Purpose:** Produce ~10 tool-calling traces, serialize them to OSFT JSONL, and validate
the schema so Farbod's team can confirm the format works with Training Hub's `osft()` call.

**Prerequisites:**
- vLLM serving Gemma 4 31B is accessible (set `VLLM_API_KEY` and `VLLM_API_BASE`)
- Langflow agent is deployed and accessible (set `LANGFLOW_API_KEY` and `LANGFLOW_URL`)
- MCP Gateway is reachable from this cluster (set `GATEWAY_URL`, `KEYCLOAK_URL`, `CLIENT_SECRET`)
- This notebook is intended to run in an RHOAI workbench pointed at the `OTel_agentic_improvements` repo

**D-19 Gate:** The dry-run output must be sent to Farbod's team. They run
`osft(data_path="dry-run.jsonl")` and confirm it tokenizes correctly and loss values are normal.
Do NOT proceed to the full production run until they confirm.

In [ ]:
import sys
import os
import json
import logging

# Add repo root to path for proper package imports
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
# If running from sdg/notebooks/, this resolves to the repo root
sys.path.insert(0, os.path.abspath('../..'))

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# ── Config Loading with env-var interpolation ──
# yaml.safe_load() does NOT expand ${VAR} placeholders.
# We walk the tree and call os.path.expandvars() on every string value.

def _expand_vars_recursive(obj):
    """Recursively apply os.path.expandvars to all string values."""
    if isinstance(obj, str):
        expanded = os.path.expandvars(obj)
        if '${' in expanded:
            logger.warning('Unexpanded env var: %r -- set the variable', expanded)
        return expanded
    elif isinstance(obj, dict):
        return {k: _expand_vars_recursive(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_expand_vars_recursive(item) for item in obj]
    return obj

def _load_config(path):
    """Load YAML config with os.path.expandvars() interpolation."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return _expand_vars_recursive(raw) if raw else {}

# Load flow config
config = _load_config('../../sdg/config/flow-training.yaml')
print('Model:', config.get('model_config', {}).get('model'))
print('API base:', config.get('model_config', {}).get('api_base'))
print('Agent framework:', config.get('agent_config', {}).get('agent_framework'))

In [ ]:
# ── Seed Materialization ──
# Ensure tool metadata seeds are materialized from the MCP Gateway
# before running the generation flow.

from sdg.scripts.materialize_seeds import (
    materialize_tool_seeds,
    acquire_jwt,
    mcp_initialize,
    fetch_tools,
)

import time
from pathlib import Path

seeds_output = Path('../../sdg/data/seeds/tool_metadata.jsonl')
seeds_config = '../../sdg/config/seed-scenarios.yaml'

# Check if seeds exist and are fresh (< 1 hour old)
need_materialize = True
if seeds_output.exists():
    age = time.time() - seeds_output.stat().st_mtime
    if age < 3600:
        count = sum(1 for line in seeds_output.open() if line.strip())
        print(f'Seeds file is fresh ({age:.0f}s old, {count} records) -- skipping materialization')
        need_materialize = False

if need_materialize:
    gateway_url = os.environ.get('GATEWAY_URL', 'https://mcp.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com/mcp')
    keycloak_url = os.environ.get('KEYCLOAK_URL', 'https://mcp-keycloak-otel25-gateway.apps.cluster-wfhjs.wfhjs.sandbox1330.opentlc.com')
    client_id = os.environ.get('CLIENT_ID', 'sdg-pipeline')
    client_secret = os.environ.get('CLIENT_SECRET', '')

    if not client_secret:
        raise ValueError('CLIENT_SECRET env var is required for seed materialization')

    jwt = acquire_jwt(keycloak_url, client_id, client_secret)
    session_id = mcp_initialize(gateway_url, jwt)
    tools = fetch_tools(gateway_url, jwt, session_id)
    count = materialize_tool_seeds(tools, seeds_config, str(seeds_output))
    print(f'Materialized {count} seed records from {len(tools)} gateway tools')

In [ ]:
# ── Load and Run Generation Flow (10 examples) ──

from sdg_hub import FlowRegistry, Flow

FlowRegistry.discover_flows()
flow_path = FlowRegistry.get_flow_path('MCP Server Distillation')
flow = Flow.from_yaml(flow_path)

# Dry-run runtime params: minimal generation
dry_run_params = {
    'multiply_tool_rows': {'num_samples': 1},
    'sample_tools': {'num_samples': 1},
}

# Load 10 seeds from single_tool tier
seeds = []
with open(seeds_output) as f:
    for line in f:
        record = json.loads(line.strip())
        if record.get('tier') == 'single_tool':
            seeds.append(record)
seeds = seeds[:10]
print(f'Using {len(seeds)} seeds for dry-run')

# Run generation
output = flow.generate(
    seed_dataset=seeds,
    model_config=config.get('model_config', {}),
    agent_config=config.get('agent_config', {}),
    runtime_params=dry_run_params,
)

# Convert to list of dicts
if hasattr(output, 'iterrows'):
    results = [row.to_dict() for _, row in output.iterrows()]
else:
    results = list(output)

print(f'Generated {len(results)} examples')

# Save raw output
dry_run_dir = Path('../../sdg/data/raw/dry-run/')
dry_run_dir.mkdir(parents=True, exist_ok=True)
with open(dry_run_dir / 'single_tool_raw.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r, default=str) + '\n')
print(f'Raw output saved to {dry_run_dir / "single_tool_raw.jsonl"}')

In [ ]:
# ── Column Mapping ──
# Bridge sdg_hub output columns to serializer input keys
# Same COLUMN_MAP as run-evaluation.py

COLUMN_MAP = {
    'question': 'question',
    'agent_text': 'answer',
    'response_text': 'answer',
    'tool_trace': 'tool_trace',
    'agent_tool_trace': 'tool_trace',
    'tool_list': 'tools',
}
REQUIRED_KEYS = {'question', 'tool_trace', 'answer', 'tools'}

mapped_rows = []
for i, row in enumerate(results):
    mapped = {}
    for src, dst in COLUMN_MAP.items():
        if src in row and (dst not in mapped or mapped[dst] is None):
            mapped[dst] = row[src]
    for key in REQUIRED_KEYS:
        if key in row and key not in mapped:
            mapped[key] = row[key]

    # Parse JSON strings
    for key in ('tool_trace', 'tools'):
        if isinstance(mapped.get(key), str):
            try:
                mapped[key] = json.loads(mapped[key])
            except (json.JSONDecodeError, ValueError):
                pass

    missing = REQUIRED_KEYS - set(mapped.keys())
    if missing:
        raise ValueError(
            f'Row {i}: missing keys {missing} after mapping. '
            f'Available: {sorted(row.keys())}'
        )
    mapped_rows.append(mapped)

print(f'Column mapping complete: {len(mapped_rows)} rows mapped')

In [ ]:
# ── OSFT Serialization ──
# Import from the proper package path (underscore filename)

from sdg.scripts.serialize_osft import serialize_single_turn

serialized_examples = []
for i, row in enumerate(mapped_rows):
    try:
        s = serialize_single_turn(row)
        serialized_examples.append(s)
    except (ValueError, KeyError) as exc:
        print(f'Row {i} serialization failed: {exc}')

print(f'Serialized {len(serialized_examples)}/{len(mapped_rows)} examples')

In [ ]:
# ── OSFT Validation ──
# Per amended D-04, OSFT validation runs automatically, same as in run-evaluation.py.
# This is NOT a separate manual step -- it validates every serialized row.

from sdg.lib.osft_validation import validate_osft_line

validated = []
failed = []
for i, example in enumerate(serialized_examples):
    errors = validate_osft_line(example)
    if errors:
        failed.append({'row': i, 'errors': errors})
        print(f'Row {i} FAILED: {errors}')
    else:
        validated.append(example)

print(f'\nOSFT Validation Results:')
print(f'  Passed: {len(validated)}/{len(serialized_examples)}')
print(f'  Failed: {len(failed)}/{len(serialized_examples)}')

if failed:
    print('\nFailed rows (first 5):')
    for f in failed[:5]:
        print(f'  Row {f["row"]}: {f["errors"]}')

# Save validated output for Farbod's team
dry_run_output = dry_run_dir / 'dry-run.jsonl'
with open(dry_run_output, 'w') as f:
    for example in validated:
        f.write(json.dumps(example) + '\n')
print(f'\nValidated output saved to {dry_run_output}')

In [ ]:
# ── Sample Output ──
# Display 3 example OSFT JSONL records for visual inspection

from IPython.display import JSON, display

for i, example in enumerate(validated[:3]):
    print(f'\n--- Example {i+1} ---')
    print(f'Messages count: {len(example["messages"])}')
    for msg in example['messages']:
        role = msg['role']
        if role == 'system':
            print(f'  [{role}] {msg["content"][:80]}...')
        elif role == 'user':
            print(f'  [{role}] {msg["content"][:120]}')
        elif role == 'assistant' and msg.get('tool_calls'):
            for tc in msg['tool_calls']:
                print(f'  [{role}] tool_call: {tc["function"]["name"]}({tc["function"]["arguments"][:60]})')
        elif role == 'tool':
            print(f'  [{role}] call_id={msg["tool_call_id"][:20]}... content={msg["content"][:60]}...')
        elif role == 'assistant':
            print(f'  [{role}] {msg["content"][:120]}')
    print()
    # Full JSON for first example
    if i == 0:
        print('Full JSON:')
        print(json.dumps(example, indent=2)[:2000])

## Next Step: Send sdg/data/raw/dry-run/dry-run.jsonl to Farbod's team for OSFT validation

Farbod's team should:
1. Run `osft(data_path="dry-run.jsonl")` with Training Hub
2. Confirm the data tokenizes correctly
3. Confirm loss values are normal

**D-19 Gate:** Do NOT proceed to the full production run (03-production-run.ipynb)
until Farbod's team confirms the format works.

If issues are found:
- If `tool_calls` not recognized: check if `mini_trainer` needs a patch or data needs pre-tokenization
- If loss values abnormal: check token masking in the chat template